In [ ]:
# ============================================================
# COMPLETE HEA-Net-WEP PIPELINE (OPTIMIZED FOR MONUSEG)
# ============================================================

import os
import glob
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from pytorch_wavelets import DWTForward
from PIL import Image
from tqdm import tqdm
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# DATASET (Modified with Cropping, Augmentation & Normalization)
# ------------------------------------------------------------
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, size=256, is_train=True):
        self.images = sorted(glob.glob(os.path.join(image_dir, "*")))
        self.masks  = sorted(glob.glob(os.path.join(mask_dir, "*")))
        self.size = size
        self.is_train = is_train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = Image.open(self.images[idx]).convert("RGB")
        mask = Image.open(self.masks[idx]).convert("L")

        if self.is_train:
            # 1. Random Crop (Preserves original resolution)
            i, j, h, w = T.RandomCrop.get_params(img, output_size=(self.size, self.size))
            img = TF.crop(img, i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)

            # 2. Random Horizontal Flip
            if random.random() > 0.5:
                img = TF.hflip(img)
                mask = TF.hflip(mask)

            # 3. Random Vertical Flip
            if random.random() > 0.5:
                img = TF.vflip(img)
                mask = TF.vflip(mask)
                
            # 4. Color Jitter (Handles H&E stain variations)
            img = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)(img)
        else:
            # Center crop for validation
            img = TF.center_crop(img, (self.size, self.size))
            mask = TF.center_crop(mask, (self.size, self.size))

        img = TF.to_tensor(img)
        mask = TF.to_tensor(mask)
        
        # 5. Normalize using ImageNet stats
        img = TF.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        mask = (mask > 0.5).float()
        return img, mask

# -------------------------------------------------------------------------
# Shannon Entropy Module
# -------------------------------------------------------------------------
class PatchEntropyAttention(nn.Module):
    def __init__(self, patch=7):
        super().__init__()
        self.patch = patch
        self.avgpool = nn.AvgPool2d(kernel_size=patch, stride=1, padding=patch//2)

    def shannon_entropy(self, x, eps=1e-8):
        x_abs = torch.abs(x) + eps
        p = x_abs / (x_abs.sum(dim=(2, 3), keepdim=True) + eps)
        H = -p * torch.log(p + eps)
        return H.sum(dim=1, keepdim=True)

    def forward(self, LL, LH, HL, HH, return_maps=False):
        H_LL = self.avgpool(self.shannon_entropy(LL))
        H_LH = self.avgpool(self.shannon_entropy(LH))
        H_HL = self.avgpool(self.shannon_entropy(HL))
        H_HH = self.avgpool(self.shannon_entropy(HH))

        weights = torch.softmax(torch.cat([H_LL, H_LH, H_HL, H_HH], dim=1), dim=1)
        w_LL, w_LH, w_HL, w_HH = torch.split(weights, 1, dim=1)

        if return_maps:
            return (
                LL * w_LL, LH * w_LH, HL * w_HL, HH * w_HH,
                H_LL, H_LH, H_HL, H_HH
            )

        return LL * w_LL, LH * w_LH, HL * w_HL, HH * w_HH
    

# ------------------------------------------------------------
# WEAM (Wavelet Enhanced Attention Module)
# ------------------------------------------------------------
class WEAM(nn.Module):
    def __init__(self, channels, wave='db6'):
        super().__init__()
        self.dwt = DWTForward(J=1, wave=wave, mode='reflect')
        self.entropy_att = PatchEntropyAttention()
        self.reduction = nn.Conv2d(channels * 4, channels, 1)
        self.gamma = nn.Parameter(torch.ones(1, channels, 1, 1))

    def spatial_attention(self, x):
        mean = x.mean(dim=(2, 3), keepdim=True)
        var  = ((x - mean)**2).mean(dim=(2, 3), keepdim=True)
        energy = (x - mean)**2 / (var + 1e-6)
        return torch.sigmoid(energy)

    def forward(self, x, return_attention=False):
        B, C, H, W = x.shape
        LL, HF = self.dwt(x)
        hf = HF[0]
        LH, HL, HH = hf[:, :, 0], hf[:, :, 1], hf[:, :, 2]

        LL, LH, HL, HH = self.entropy_att(LL, LH, HL, HH)

        combined = torch.cat([LL, LH, HL, HH], dim=1)
        out = self.reduction(combined)
        
        out = F.interpolate(out, (H, W), mode='bilinear', align_corners=False)

        att = self.spatial_attention(out)

        if return_attention:
            return out * att * self.gamma, att

        return out * att * self.gamma

# -------------------------------------------------------------------------
# PAC block (Polar Attention Coordinate)
# -------------------------------------------------------------------------
class PAC(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.radial = nn.Conv2d(channels, channels, 1)
        self.angular = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        yy, xx = torch.meshgrid(torch.arange(H, device=x.device), 
                                torch.arange(W, device=x.device), indexing='ij')
        
        yy = yy - H/2
        xx = xx - W/2

        r = torch.sqrt(xx**2 + yy**2)
        r = r / (r.max() + 1e-6)
        t = torch.atan2(yy, xx)
        t = (t + math.pi) / (2 * math.pi)

        r = r.unsqueeze(0).unsqueeze(0)
        t = t.unsqueeze(0).unsqueeze(0)

        return x + self.radial(x * r) + self.angular(x * t)

# ------------------------------------------------------------
# VISUALIZE PAC KERNELS
# ------------------------------------------------------------
def visualize_pac_kernels(model, img):
    model.eval()

    img = img.to(device)

    with torch.no_grad():

        # Encoder feature extraction
        f1 = model.enc1(img)
        e1 = f1 + model.weam1(f1)

        f2 = model.enc2(model.pool(e1))
        e2 = f2 + model.weam2(f2)

        f3 = model.enc3(model.pool(e2))
        e3 = f3 + model.weam3(f3)

        bottleneck = model.bottleneck(model.pool(e3))

        B, C, H, W = bottleneck.shape

        yy, xx = torch.meshgrid(
            torch.arange(H, device=device),
            torch.arange(W, device=device),
            indexing='ij'
        )

        yy = yy - H / 2
        xx = xx - W / 2

        # Radial kernel
        r = torch.sqrt(xx**2 + yy**2)
        r = r / (r.max() + 1e-6)

        # Angular kernel
        theta = torch.atan2(yy, xx)
        theta = (theta + math.pi) / (2 * math.pi)

        combined_kernel = (r + theta) / 2

        radial_response = bottleneck[0, 0].cpu() * r.cpu()
        angular_response = bottleneck[0, 0].cpu() * theta.cpu()

        before = bottleneck[0].mean(dim=0).cpu()
        after = model.mixer(bottleneck)[0].mean(dim=0).cpu()
        difference = after - before

        before_edges = cv2.Canny(
        ((before.numpy() - before.min()) / (before.max()-before.min()+1e-8) * 255).astype(np.uint8),
        50, 150
        )

        after_edges = cv2.Canny(
        ((after.numpy() - after.min()) / (after.max()-after.min()+1e-8) * 255).astype(np.uint8),
        50, 150
        )

    plt.figure(figsize=(18, 10))

    # Original bottleneck feature
    plt.subplot(2, 3, 1)
    plt.imshow(before, cmap='gray')
    plt.title("Before PAC")
    plt.colorbar()
    plt.axis('off')

    # Radial kernel
    plt.subplot(2, 3, 2)
    plt.imshow(r.cpu(), cmap='viridis')
    plt.title("Radial Kernel (r)")
    plt.colorbar()
    plt.axis('off')

    # Angular kernel
    plt.subplot(2, 3, 3)
    plt.imshow(theta.cpu(), cmap='hsv')
    plt.title("Angular Kernel (θ)")
    plt.colorbar()
    plt.axis('off')

    # Radial response
    plt.subplot(2, 3, 4)
    plt.imshow(radial_response, cmap='inferno')
    plt.title("Radial Attention Response")
    plt.colorbar()
    plt.axis('off')

    # Angular response
    plt.subplot(2, 3, 5)
    plt.imshow(angular_response, cmap='inferno')
    plt.title("Angular Attention Response")
    plt.colorbar()
    plt.axis('off')

    # Final PAC output
    plt.subplot(2, 3, 6)
    plt.imshow(after, cmap='gray')
    plt.title("After PAC")
    plt.colorbar()
    plt.axis('off')

    # Structural Difference Map
    plt.subplot(2,3,7)
    im = plt.imshow(difference, cmap='bwr')
    plt.title("PAC Structural Change")
    plt.axis('off')

    #Edge Visualization
    plt.subplot(2,3,8)
    plt.imshow(after_edges - before_edges, cmap='hot')
    plt.title("Boundary Enhancement")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

def visualize_polar_coordinates(H=256, W=256):
        yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')

        yy = yy - H/2
        xx = xx - W/2

        r = np.sqrt(xx**2 + yy**2)
        r = r / (r.max() + 1e-6)

        theta = np.arctan2(yy, xx)
        theta = (theta + np.pi) / (2*np.pi)

        plt.figure(figsize=(12,5))

        plt.subplot(1,2,1)
        plt.imshow(r, cmap='viridis')
        plt.colorbar()
        plt.title("Radial Component (r)")
        plt.axis('off')

        plt.subplot(1,2,2)
        plt.imshow(theta, cmap='hsv')
        plt.colorbar()
        plt.title("Angular Component (θ)")
        plt.axis('off')

        plt.show()

def visualize_pac_effect(model, img):
    model.eval()
    img = img.to(device)

    with torch.no_grad():
        f1 = model.enc1(img)
        e1 = f1 + model.weam1(f1)

        f2 = model.enc2(model.pool(e1))
        e2 = f2 + model.weam2(f2)

        f3 = model.enc3(model.pool(e2))
        e3 = f3 + model.weam3(f3)

        before = model.bottleneck(model.pool(e3))
        after = model.mixer(before)

    plt.figure(figsize=(10,4))

    plt.subplot(1,2,1)
    plt.imshow(before[0,0].cpu(), cmap='gray')
    plt.colorbar()
    plt.title("Before PAC")

    plt.subplot(1,2,2)
    plt.imshow(after[0,0].cpu(), cmap='gray')
    plt.colorbar()
    plt.title("After PAC")

    plt.show()

# ------------------------------------------------------------
# BASIC CONV BLOCK (Modified to InstanceNorm)
# ------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

# ------------------------------------------------------------
# HEA-NET-WEP ARCHITECTURE
# ------------------------------------------------------------
class HEANetWEP(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, 64)
        self.weam1 = WEAM(64)
        self.enc2 = ConvBlock(64, 128)
        self.weam2 = WEAM(128)
        self.enc3 = ConvBlock(128, 256)
        self.weam3 = WEAM(256)
        
        self.pool = nn.MaxPool2d(2)
        
        self.bottleneck = ConvBlock(256, 512)
        self.mixer = PAC(512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = ConvBlock(256 + 256, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = ConvBlock(128 + 128, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = ConvBlock(64 + 64, 64)
        
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        f1 = self.enc1(x)
        e1 = f1 + self.weam1(f1)
        f2 = self.enc2(self.pool(e1))
        e2 = f2 + self.weam2(f2)
        f3 = self.enc3(self.pool(e2))
        e3 = f3 + self.weam3(f3)
        
        b = self.mixer(self.bottleneck(self.pool(e3)))
        
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        
        return self.final(d1)
    
def visualize_entropy_maps(model, img):
    model.eval()
    img = img.to(device)

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    with torch.no_grad():
        LL, HF = model.weam1.dwt(img)
        hf = HF[0]

        LH, HL, HH = hf[:, :, 0], hf[:, :, 1], hf[:, :, 2]

        outputs = model.weam1.entropy_att(
            LL, LH, HL, HH,
            return_maps=True
        )

        LL_w, LH_w, HL_w, HH_w, H_LL, H_LH, H_HL, H_HH = outputs

    maps = [H_LL, H_LH, H_HL, H_HH]
    titles = ["LL Entropy", "LH Entropy", "HL Entropy", "HH Entropy"]

    img_np = img[0].permute(1, 2, 0).cpu().numpy()
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    fig, axes = plt.subplots(1, 5, figsize=(18,5))

    # Original image
    axes[0].imshow(img_np)
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    # Entropy maps
    for i in range(4):
        im = axes[i+1].imshow(maps[i][0,0].cpu(), cmap='hot')
        axes[i+1].set_title(titles[i])
        axes[i+1].axis('off')

    # ONE shared colorbar
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# GRAD-CAM
# ------------------------------------------------------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.forward_hook)
        target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, x):
        self.model.zero_grad()

        # Forward pass
        output = self.model(x)

        # Segmentation-focused loss
        loss = output[:, 0].sum()

        # Backward pass
        loss.backward()

        grads = self.gradients
        acts = self.activations

        # Global Average Pooling on gradients
        weights = grads.mean(dim=(2, 3), keepdim=True)

        # Weighted combination
        cam = (weights * acts).sum(dim=1, keepdim=True)

        # ReLU
        cam = F.relu(cam)

        # Resize to input image size
        cam = F.interpolate(
            cam,
            size=x.shape[2:],
            mode='bilinear',
            align_corners=False
        )

        # Normalize CAM
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam


# ------------------------------------------------------------
# RUN GRAD-CAM VISUALIZATION
# ------------------------------------------------------------
def run_gradcam_weam(model, img, save_path=None):
    model.eval()

    # Better semantic target layer
    cam_generator = GradCAM(model, model.bottleneck)

    img = img.to(device)

    with torch.enable_grad():
        heatmap = cam_generator.generate(img)

    # Convert heatmap
    heatmap = heatmap[0, 0].detach().cpu().numpy()

    # Convert image back from normalization
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    img_np = img[0].permute(1, 2, 0).detach().cpu().numpy()
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Plot
    plt.figure(figsize=(12, 5))

    # Original Image
    plt.subplot(1, 2, 1)
    plt.imshow(img_np)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img_np)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    plt.colorbar()
    plt.title("Grad-CAM Attention Map")
    plt.axis('off')

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

# ------------------------------------------------------------
# SAVE GRAD-CAM FOR ALL TEST IMAGES
# ------------------------------------------------------------
def save_gradcam_for_testset(model, loader, save_dir="gradcam_outputs"):
    os.makedirs(save_dir, exist_ok=True)

    model.eval()

    cam_generator = GradCAM(model, model.bottleneck)

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    img_counter = 0

    for imgs, masks in tqdm(loader, desc="Generating Grad-CAMs"):

        imgs = imgs.to(device)

        for i in range(imgs.size(0)):

            img = imgs[i].unsqueeze(0)

            with torch.enable_grad():
                heatmap = cam_generator.generate(img)

            heatmap = heatmap[0, 0].detach().cpu().numpy()

            # Unnormalize image
            img_np = img[0].permute(1, 2, 0).detach().cpu().numpy()
            img_np = std * img_np + mean
            img_np = np.clip(img_np, 0, 1)

            fig, axes = plt.subplots(1, 2, figsize=(10,5))

            # Original image
            axes[0].imshow(img_np)
            axes[0].set_title("Original Image")
            axes[0].axis('off')

            # Grad-CAM overlay
            im = axes[1].imshow(img_np)
            im = axes[1].imshow(heatmap, cmap='jet', alpha=0.5)

            axes[1].set_title("Grad-CAM")
            axes[1].axis('off')

            # SINGLE shared colorbar
            fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04)


            plt.tight_layout()

            save_path = os.path.join(
                save_dir,
                f"gradcam_{img_counter}.png"
            )

            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()

            img_counter += 1

    print(f"All Grad-CAM images saved in: {save_dir}")
    
# ------------------------------------------------------------
# LOSS + METRICS
# ------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def combined_loss(pred, target):
    return criterion(pred, target) + dice_loss(pred, target)

def compute_metrics(pred, target):
    pred = torch.sigmoid(pred)
    p = (pred > 0.5).float()
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - inter
    dice = (2 * inter) / (p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6)
    iou  = inter / (union + 1e-6)
    return dice.mean().item(), iou.mean().item()

# ------------------------------------------------------------
# TRAINING FUNCTION
# ------------------------------------------------------------
def train_model(train_loader, val_loader, epochs=100, lr=1e-3):
    model = HEANetWEP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=10, factor=0.5, verbose=True)

    best_dice = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for img, mask in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img, mask = img.to(device), mask.to(device)

            pred = model(img)
            loss = combined_loss(pred, mask)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_dice, val_iou = 0, 0

        with torch.no_grad():
            for img, mask in val_loader:
                img, mask = img.to(device), mask.to(device)
                pred = model(img)
                d, i = compute_metrics(pred, mask)
                val_dice += d
                val_iou += i

        val_dice /= len(val_loader)
        val_iou /= len(val_loader)
        
        scheduler.step(val_dice)

        print(f"Train Loss: {train_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), "best_model.pth")
            print(">>> Best model saved! <<<")

    print("Training Complete. Best Dice:", best_dice)
    return model

# ------------------------------------------------------------
# TESTING / VISUALIZATION FUNCTION
# ------------------------------------------------------------
def save_prediction_visualizations(model, loader, save_dir="saved_predictions"):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    batch_idx = 0

    # ImageNet stats for un-normalizing
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Saving Predictions"):
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            preds_probs = torch.sigmoid(preds)

            imgs_np = imgs.cpu().numpy()
            masks_np = masks.cpu().numpy()
            preds_np = preds_probs.cpu().numpy()

            batch_size = imgs_np.shape[0]

            for i in range(batch_size):
                img = np.transpose(imgs_np[i], (1, 2, 0))
                
                # Un-normalize back to 0-1 range for plotting
                img = std * img + mean
                img = np.clip(img, 0, 1)

                true_mask = masks_np[i, 0]
                pred_binary = (preds_np[i, 0] > 0.5).astype(np.uint8)

                img_uint8 = (img * 255).astype(np.uint8)
                overlay_img = img_uint8.copy()

                try:
                    contours, _ = cv2.findContours(
                        pred_binary,
                        cv2.RETR_EXTERNAL,
                        cv2.CHAIN_APPROX_SIMPLE
                    )
                    cv2.drawContours(overlay_img, contours, -1, (0, 255, 0), 1)
                except Exception:
                    pass 

                fig, axes = plt.subplots(1, 4, figsize=(20, 5))

                axes[0].imshow(img)
                axes[0].set_title("Original Image (Cropped)")
                axes[0].axis("off")

                axes[1].imshow(true_mask, cmap="gray")
                axes[1].set_title("Ground Truth")
                axes[1].axis("off")

                axes[2].imshow(pred_binary, cmap="gray")
                axes[2].set_title("Predicted Mask")
                axes[2].axis("off")

                axes[3].imshow(overlay_img)
                axes[3].set_title("Overlay")
                axes[3].axis("off")

                plt.tight_layout()
                save_path = os.path.join(save_dir, f"batch_{batch_idx}_img_{i}.png")
                plt.savefig(save_path, dpi=300)
                plt.close(fig)

            batch_idx += 1

    print(f"All predictions saved successfully in: {save_dir}")

# ============================================================
# MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    # UPDATE THESE PATHS TO MATCH YOUR SYSTEM
    train_images = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Training/TissueImages"
    train_masks  = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Training/GroundTruth"
    val_images   = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Test/TissueImages"
    val_masks    = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Test/GroundTruth"

    # 1. Initialize Dataloaders (Note the is_train flags)
    print("Loading datasets...")
    train_loader = DataLoader(SegmentationDataset(train_images, train_masks, size=256, is_train=True), batch_size=4, shuffle=True)
    val_loader   = DataLoader(SegmentationDataset(val_images, val_masks, size=256, is_train=False), batch_size=4, shuffle=False)

    # 2. Train the Model
    print(f"Starting training on {device}...")
    # Kept to 300 epochs, but scheduler will drop LR if it plateaus
    model = train_model(train_loader, val_loader, epochs=200, lr=1e-3)

    # 3. Load Best Weights for Visualization
    print("Loading best model weights for testing...")
    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    sample_img, _ = next(iter(val_loader))

    # Take 1 image
    sample_img = sample_img[0].unsqueeze(0)

visualize_entropy_maps(model, sample_img)

save_gradcam_for_testset(
    model,
    val_loader,
    save_dir="all_gradcam_results"
)

visualize_polar_coordinates()
visualize_pac_effect(model, sample_img)
visualize_pac_kernels(model, sample_img)

    # 4. Save Predictions
    print("Generating visualizations...")
    save_prediction_visualizations(model, val_loader, save_dir="saved_predictions")

Loading datasets...
Starting training on cuda...


Epoch 1/200: 100%|████████████████████████████████| 8/8 [00:02<00:00,  3.66it/s]


Train Loss: 0.9228 | Val Dice: 0.6973 | Val IoU: 0.5394
>>> Best model saved! <<<


Epoch 2/200: 100%|████████████████████████████████| 8/8 [00:02<00:00,  3.75it/s]


Train Loss: 0.7565 | Val Dice: 0.7531 | Val IoU: 0.6060
>>> Best model saved! <<<


...

Train Loss: 0.4264 | Val Dice: 0.8235 | Val IoU: 0.7016


Epoch 200/200: 100%|██████████████████████████████| 8/8 [00:02<00:00,  3.76it/s]
/tmp/ipykernel_4290/3322691312.py:396: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.l

Train Loss: 0.4015 | Val Dice: 0.8234 | Val IoU: 0.7015
Training Complete. Best Dice: 0.8255063593387604
Loading best model weights for testing...
Generating visualizations...


Saving Predictions: 100%|█████████████████████████| 4/4 [00:12<00:00,  3.21s/it]

All predictions saved successfully in: saved_predictions


In [1]:
# ============================================================
# COMPLETE HEA-Net-WEP PIPELINE (OPTIMIZED FOR MONUSEG)
# ============================================================

import os
import glob
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from pytorch_wavelets import DWTForward
from PIL import Image
from tqdm import tqdm
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# DATASET (Modified with Cropping, Augmentation & Normalization)
# ------------------------------------------------------------
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, size=256, is_train=True):
        self.images = sorted(glob.glob(os.path.join(image_dir, "*")))
        self.masks  = sorted(glob.glob(os.path.join(mask_dir, "*")))
        self.size = size
        self.is_train = is_train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img  = Image.open(self.images[idx]).convert("RGB")
        mask = Image.open(self.masks[idx]).convert("L")

        if self.is_train:
            # 1. Random Crop (Preserves original resolution)
            i, j, h, w = T.RandomCrop.get_params(img, output_size=(self.size, self.size))
            img = TF.crop(img, i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)

            # 2. Random Horizontal Flip
            if random.random() > 0.5:
                img = TF.hflip(img)
                mask = TF.hflip(mask)

            # 3. Random Vertical Flip
            if random.random() > 0.5:
                img = TF.vflip(img)
                mask = TF.vflip(mask)
                
            # 4. Color Jitter (Handles H&E stain variations)
            img = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)(img)
        else:
            # Center crop for validation
            img = TF.center_crop(img, (self.size, self.size))
            mask = TF.center_crop(mask, (self.size, self.size))

        img = TF.to_tensor(img)
        mask = TF.to_tensor(mask)
        
        # 5. Normalize using ImageNet stats
        img = TF.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        mask = (mask > 0.5).float()
        return img, mask

# -------------------------------------------------------------------------
# Shannon Entropy Module
# -------------------------------------------------------------------------
class PatchEntropyAttention(nn.Module):
    def __init__(self, patch=7):
        super().__init__()
        self.patch = patch
        self.avgpool = nn.AvgPool2d(kernel_size=patch, stride=1, padding=patch//2)

    def shannon_entropy(self, x, eps=1e-8):
        x_abs = torch.abs(x) + eps
        p = x_abs / (x_abs.sum(dim=(2, 3), keepdim=True) + eps)
        H = -p * torch.log(p + eps)
        return H.sum(dim=1, keepdim=True)

    def forward(self, LL, LH, HL, HH):
        H_LL = self.avgpool(self.shannon_entropy(LL))
        H_LH = self.avgpool(self.shannon_entropy(LH))
        H_HL = self.avgpool(self.shannon_entropy(HL))
        H_HH = self.avgpool(self.shannon_entropy(HH))

        weights = torch.softmax(torch.cat([H_LL, H_LH, H_HL, H_HH], dim=1), dim=1)
        w_LL, w_LH, w_HL, w_HH = torch.split(weights, 1, dim=1)

        return LL * w_LL, LH * w_LH, HL * w_HL, HH * w_HH

# ------------------------------------------------------------
# WEAM (Wavelet Enhanced Attention Module)
# ------------------------------------------------------------
class WEAM(nn.Module):
    def __init__(self, channels, wave='db6'):
        super().__init__()
        self.dwt = DWTForward(J=1, wave=wave, mode='reflect')
        self.entropy_att = PatchEntropyAttention()
        self.reduction = nn.Conv2d(channels * 4, channels, 1)
        self.gamma = nn.Parameter(torch.ones(1, channels, 1, 1))

    def spatial_attention(self, x):
        mean = x.mean(dim=(2, 3), keepdim=True)
        var  = ((x - mean)**2).mean(dim=(2, 3), keepdim=True)
        energy = (x - mean)**2 / (var + 1e-6)
        return torch.sigmoid(energy)

    def forward(self, x):
        B, C, H, W = x.shape
        LL, HF = self.dwt(x)
        hf = HF[0]
        LH, HL, HH = hf[:, :, 0], hf[:, :, 1], hf[:, :, 2]

        LL, LH, HL, HH = self.entropy_att(LL, LH, HL, HH)

        combined = torch.cat([LL, LH, HL, HH], dim=1)
        out = self.reduction(combined)
        
        out = F.interpolate(out, (H, W), mode='bilinear', align_corners=False)
        out = out * self.spatial_attention(out)
        
        return out * self.gamma

# -------------------------------------------------------------------------
# PAC block (Polar Attention Coordinate)
# -------------------------------------------------------------------------
class PAC(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.radial = nn.Conv2d(channels, channels, 1)
        self.angular = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        yy, xx = torch.meshgrid(torch.arange(H, device=x.device), 
                                torch.arange(W, device=x.device), indexing='ij')
        
        yy = yy - H/2
        xx = xx - W/2

        r = torch.sqrt(xx**2 + yy**2)
        r = r / (r.max() + 1e-6)
        t = torch.atan2(yy, xx)
        t = (t + math.pi) / (2 * math.pi)

        r = r.unsqueeze(0).unsqueeze(0)
        t = t.unsqueeze(0).unsqueeze(0)

        return x + self.radial(x * r) + self.angular(x * t)

# ------------------------------------------------------------
# BASIC CONV BLOCK (Modified to InstanceNorm)
# ------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

# ------------------------------------------------------------
# HEA-NET-WEP ARCHITECTURE
# ------------------------------------------------------------
class HEANetWEP(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, 64)
        self.weam1 = WEAM(64)
        self.enc2 = ConvBlock(64, 128)
        self.weam2 = WEAM(128)
        self.enc3 = ConvBlock(128, 256)
        self.weam3 = WEAM(256)
        
        self.pool = nn.MaxPool2d(2)
        
        self.bottleneck = ConvBlock(256, 512)
        self.mixer = PAC(512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = ConvBlock(256 + 256, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = ConvBlock(128 + 128, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = ConvBlock(64 + 64, 64)
        
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        f1 = self.enc1(x)
        e1 = f1 + self.weam1(f1)
        f2 = self.enc2(self.pool(e1))
        e2 = f2 + self.weam2(f2)
        f3 = self.enc3(self.pool(e2))
        e3 = f3 + self.weam3(f3)
        
        b = self.mixer(self.bottleneck(self.pool(e3)))
        
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        
        return self.final(d1)
# ------------------------------------------------------------
# LOSS + METRICS
# ------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def combined_loss(pred, target):
    return criterion(pred, target) + dice_loss(pred, target)

def compute_metrics(pred, target):
    pred = torch.sigmoid(pred)
    p = (pred > 0.5).float()
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - inter
    dice = (2 * inter) / (p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1e-6)
    iou  = inter / (union + 1e-6)
    return dice.mean().item(), iou.mean().item()

# ------------------------------------------------------------
# TRAINING FUNCTION
# ------------------------------------------------------------
def train_model(train_loader, val_loader, epochs=100, lr=1e-3):
    model = HEANetWEP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=10, factor=0.5, verbose=True)

    best_dice = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for img, mask in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            img, mask = img.to(device), mask.to(device)

            pred = model(img)
            loss = combined_loss(pred, mask)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_dice, val_iou = 0, 0

        with torch.no_grad():
            for img, mask in val_loader:
                img, mask = img.to(device), mask.to(device)
                pred = model(img)
                d, i = compute_metrics(pred, mask)
                val_dice += d
                val_iou += i

        val_dice /= len(val_loader)
        val_iou /= len(val_loader)
        
        scheduler.step(val_dice)

        print(f"Train Loss: {train_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), "best_model.pth")
            print(">>> Best model saved! <<<")

    print("Training Complete. Best Dice:", best_dice)
    return model

# ------------------------------------------------------------
# TESTING / VISUALIZATION FUNCTION
# ------------------------------------------------------------
def save_prediction_visualizations(model, loader, save_dir="saved_predictions"):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    batch_idx = 0

    # ImageNet stats for un-normalizing
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Saving Predictions"):
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            preds_probs = torch.sigmoid(preds)

            imgs_np = imgs.cpu().numpy()
            masks_np = masks.cpu().numpy()
            preds_np = preds_probs.cpu().numpy()

            batch_size = imgs_np.shape[0]

            for i in range(batch_size):
                img = np.transpose(imgs_np[i], (1, 2, 0))
                
                # Un-normalize back to 0-1 range for plotting
                img = std * img + mean
                img = np.clip(img, 0, 1)

                true_mask = masks_np[i, 0]
                pred_binary = (preds_np[i, 0] > 0.5).astype(np.uint8)

                img_uint8 = (img * 255).astype(np.uint8)
                overlay_img = img_uint8.copy()

                try:
                    contours, _ = cv2.findContours(
                        pred_binary,
                        cv2.RETR_EXTERNAL,
                        cv2.CHAIN_APPROX_SIMPLE
                    )
                    cv2.drawContours(overlay_img, contours, -1, (0, 255, 0), 1)
                except Exception:
                    pass 

                fig, axes = plt.subplots(1, 4, figsize=(20, 5))

                axes[0].imshow(img)
                axes[0].set_title("Original Image (Cropped)")
                axes[0].axis("off")

                axes[1].imshow(true_mask, cmap="gray")
                axes[1].set_title("Ground Truth")
                axes[1].axis("off")

                axes[2].imshow(pred_binary, cmap="gray")
                axes[2].set_title("Predicted Mask")
                axes[2].axis("off")

                axes[3].imshow(overlay_img)
                axes[3].set_title("Overlay")
                axes[3].axis("off")

                plt.tight_layout()
                save_path = os.path.join(save_dir, f"batch_{batch_idx}_img_{i}.png")
                plt.savefig(save_path, dpi=300)
                plt.close(fig)

            batch_idx += 1

    print(f"All predictions saved successfully in: {save_dir}")

# ============================================================
# MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    # UPDATE THESE PATHS TO MATCH YOUR SYSTEM
    train_images = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Training/TissueImages"
    train_masks  = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Training/GroundTruth"
    val_images   = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Test/TissueImages"
    val_masks    = "/home/nit/Lopa/Segmentation/Monuseg_segmentation/MonuSeg/Test/GroundTruth"

    # 1. Initialize Dataloaders (Note the is_train flags)
    print("Loading datasets...")
    train_loader = DataLoader(SegmentationDataset(train_images, train_masks, size=256, is_train=True), batch_size=8, shuffle=True)
    val_loader   = DataLoader(SegmentationDataset(val_images, val_masks, size=256, is_train=False), batch_size=8, shuffle=False)

    # 2. Train the Model
    print(f"Starting training on {device}...")
    # Kept to 300 epochs, but scheduler will drop LR if it plateaus
    model = train_model(train_loader, val_loader, epochs=200, lr=1e-3)

    # 3. Load Best Weights for Visualization
    print("Loading best model weights for testing...")
    model.load_state_dict(torch.load("best_model.pth", map_location=device))

    # 4. Save Predictions
    print("Generating visualizations...")
    save_prediction_visualizations(model, val_loader, save_dir="saved_predictions")

/home/nit/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Loading datasets...
Starting training on cuda...


Epoch 1/200: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.46it/s]


Train Loss: 0.9757 | Val Dice: 0.6880 | Val IoU: 0.5294
>>> Best model saved! <<<


Epoch 2/200: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.93it/s]


...

Train Loss: 0.4387 | Val Dice: 0.8184 | Val IoU: 0.6943


Epoch 200/200: 100%|████████████████████████████████████████████| 4/4 [00:02<00:00,  1.90it/s]
/tmp/ipykernel_155969/3751845607.py:395: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fe

Train Loss: 0.4222 | Val Dice: 0.8184 | Val IoU: 0.6943
Training Complete. Best Dice: 0.8241073191165924
Loading best model weights for testing...
Generating visualizations...


Saving Predictions: 100%|███████████████████████████████████████| 2/2 [00:13<00:00,  6.91s/it]

All predictions saved successfully in: saved_predictions
